## Camera

In [ ]:
import os
import re

dir = 'images/camera'
for filename in os.listdir(dir):
    if re.match(r'face_(\d+)\.jpg$', filename):
        number = int(re.search(r'face_(\d+)\.jpg$', filename).group(1))
        
        # Remove if even numbered
        if number % 2 == 0:
            os.remove(f'{dir}/{filename}')
            print(f"Removed: {filename}")
        else:
            print(f"Kept: {filename}")

## Image count

In [ ]:
from pathlib import Path

# Define base directory
base_dir = Path("cleaned")

# Define valid image extensions
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

# Count all image files with valid extensions
image_count = sum(1 for f in base_dir.rglob("*") if f.suffix.lower() in image_extensions)

print(f"Total image count: {image_count}")

In [ ]:
import os

root_dir = 'cleaned'

total_people = 0
total_images = 0
images_per_person = {}

for person_name in os.listdir(root_dir):
    person_path = os.path.join(root_dir, person_name)
    if os.path.isdir(person_path):
        total_people += 1
        image_files = [f for f in os.listdir(person_path)
                       if os.path.isfile(os.path.join(person_path, f))]
        image_count = len(image_files)
        total_images += image_count
        images_per_person[person_name] = image_count

sorted_people = sorted(images_per_person.items(), key=lambda x: x[1], reverse=True)

print(f'Total number of people: {total_people}')
print(f'Total number of images: {total_images}')
print('Image count per person:')
for person, count in sorted_people:
    print(f'  {person}: {count}')

In [ ]:
empty_folders = [p for p, c in images_per_person.items() if c == 0]
if empty_folders:
    print('Empty folders:')
    for person in empty_folders:
        print(f'  {person}')

In [ ]:
import hashlib

hashes = {}
duplicate_count = 0

for person, count in images_per_person.items():
    person_path = os.path.join(root_dir, person)
    for img in os.listdir(person_path):
        img_path = os.path.join(person_path, img)
        if os.path.isfile(img_path):
            with open(img_path, 'rb') as f:
                file_hash = hashlib.md5(f.read()).hexdigest()
            if file_hash in hashes:
                duplicate_count += 1
            else:
                hashes[file_hash] = img_path

print(f'Duplicate count: {duplicate_count}')

In [ ]:
from PIL import Image
import numpy as np

sizes = []

for person in os.listdir(root_dir):
    person_path = os.path.join(root_dir, person)
    if os.path.isdir(person_path):
        for img in os.listdir(person_path):
            img_path = os.path.join(person_path, img)
            try:
                with Image.open(img_path) as im:
                    sizes.append(im.size)
            except Exception:
                continue

widths, heights = zip(*sizes)
print(f'Image size:')
print(f'  Min size: {min(widths)}x{min(heights)}')
print(f'  Max size: {max(widths)}x{max(heights)}')
print(f'  Average sizw: {int(np.mean(widths))}x{int(np.mean(heights))}')

In [ ]:
if total_people > 1:
    max_count = max(images_per_person.values())
    min_count = min(images_per_person.values())
    print(f'Disbalance (max/min): {max_count} / {min_count} = {round(max_count / min_count, 2)}')

In [ ]:
import matplotlib.pyplot as plt

names = [x[0] for x in sorted_people[:30]]
counts = [x[1] for x in sorted_people[:30]]

plt.figure(figsize=(12, 6))
plt.barh(names[::-1], counts[::-1])
plt.title("Image count")
plt.xlabel("Image count")
plt.tight_layout()
plt.show()

# Retina Face crop extraction

In [ ]:
import os
import re

# Directories where odd-numbered files must be removed
even_dirs = ["images/camera"]

# Regex to extract number from filename (face_XXXX.jpg)
pattern = re.compile(r"face_(\d+)\.jpg")

def clean_directory(directory, remove_odd=True):
    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            number = int(match.group(1))
            if remove_odd and number % 2 == 1:
                os.remove(os.path.join(directory, filename))
                print(f"Removed (odd): {filename}")
            elif not remove_odd and number % 2 == 0:
                os.remove(os.path.join(directory, filename))
                print(f"Removed (even): {filename}")


for d in even_dirs:
    clean_directory(d, remove_odd=False)

In [ ]:
import os
from pathlib import Path
import shutil

# Set your source and destination
SRC_ROOT = Path("dataset-original")
DEST_DIR = Path("dataset")
DEST_DIR.mkdir(exist_ok=True)

# Initialize counter for naming
counter = 1

# List all 5 subdirectories
src_dirs = [
    "images/camera"
]

# Accepted image extensions
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for folder in src_dirs:
    src_path = SRC_ROOT / folder
    if not src_path.exists():
        print(f"Skipping missing folder: {src_path}")
        continue

    for file in sorted(src_path.iterdir()):
        if not file.is_file():
            continue
        if file.suffix.lower() not in valid_exts:
            continue

        # Create new filename like Face_1.jpg
        new_filename = f"Face_{counter}{file.suffix.lower()}"
        dest_path = DEST_DIR / new_filename

        # Move and rename
        shutil.move(str(file), str(dest_path))
        print(f"Moved: {file.name} → {new_filename}")
        counter += 1

print("done")

In [ ]:
from retinaface import RetinaFace

resp = RetinaFace.detect_faces("dataset/Face_1.jpg")
resp

In [ ]:
from pathlib import Path

# Replace with your directory
folder = Path("dataset")

# Count only files (not subdirectories)
num_files = sum(1 for f in folder.iterdir() if f.is_file())

print(f"Number of original files: {num_files}")

In [ ]:
import os
from retinaface import RetinaFace
import cv2
from pathlib import Path

# --- CONFIG ---
INPUT_DIR = Path("dataset")
OUTPUT_DIR = Path("face_crops")
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
OUTPUT_SIZE = (112, 112)

# Initialize crop index based on existing files
existing = list(OUTPUT_DIR.glob("Face_*.jpg"))
crop_index = max([int(f.stem.split("_")[1]) for f in existing if f.stem.split("_")[1].isdigit()], default=0) + 1

for image_path in sorted(INPUT_DIR.iterdir()):
    if image_path.suffix.lower() not in IMAGE_EXTS:
        continue

    print(f"Processing: {image_path.name}")

    img = cv2.imread(str(image_path))
    if img is None:
        print(f"Failed to read image: {image_path}")
        continue

    try:
        resp = RetinaFace.detect_faces(str(image_path))
    except Exception as e:
        print(f"Detection error in {image_path.name}: {e}")
        continue

    if not isinstance(resp, dict):
        print("No face detected.")
        continue

    for face_id, face_data in resp.items():
        x1, y1, x2, y2 = face_data["facial_area"]

        # Ensure coordinates are valid
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, img.shape[1]), min(y2, img.shape[0])

        face_crop = img[y1:y2, x1:x2]

        if face_crop.size == 0:
            print("Empty crop, skipping.")
            continue

        # Resize to 112x112
        face_crop_resized = cv2.resize(face_crop, OUTPUT_SIZE)

        crop_name = f"Face_{crop_index}.jpg"
        crop_path = OUTPUT_DIR / crop_name

        cv2.imwrite(str(crop_path), face_crop_resized)
        print(f"Saved: {crop_path.name}")
        crop_index += 1

print("done")

In [ ]:
from pathlib import Path

def get_already_processed_files(log_path: Path) -> set[str]:
    processed_files = set()
    with open(log_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith("Processing: "):
                filename = line.strip().split("Processing: ")[1]
                processed_files.add(filename)
    return processed_files

# Usage
already_processed = get_already_processed_files(Path("log.txt"))

In [ ]:
len(already_processed)

In [ ]:
import os
from retinaface import RetinaFace
import cv2
from pathlib import Path

# --- CONFIG ---
INPUT_DIR = Path("dataset")
OUTPUT_DIR = Path("face_crops")
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
OUTPUT_SIZE = (112, 112)
ALREADY_PROCESSED = get_already_processed_files(Path("log.txt"))

# Initialize crop index based on existing files
existing = list(OUTPUT_DIR.glob("Face_*.jpg"))
crop_index = max([int(f.stem.split("_")[1]) for f in existing if f.stem.split("_")[1].isdigit()], default=0) + 1

for image_path in sorted(INPUT_DIR.iterdir()):
    if image_path.name in ALREADY_PROCESSED:
        print(f"Skipping (already processed): {image_path.name}")
        continue

    if image_path.suffix.lower() not in IMAGE_EXTS:
        continue

    print(f"Processing: {image_path.name}")

    img = cv2.imread(str(image_path))
    if img is None:
        print(f"Failed to read image: {image_path}")
        continue

    try:
        resp = RetinaFace.detect_faces(str(image_path))
    except Exception as e:
        print(f"Detection error in {image_path.name}: {e}")
        continue

    if not isinstance(resp, dict):
        print("No face detected.")
        continue

    for face_id, face_data in resp.items():
        x1, y1, x2, y2 = face_data["facial_area"]

        # Ensure coordinates are valid
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, img.shape[1]), min(y2, img.shape[0])

        face_crop = img[y1:y2, x1:x2]

        if face_crop.size == 0:
            print("Empty crop, skipping.")
            continue

        # Resize to 112x112
        face_crop_resized = cv2.resize(face_crop, OUTPUT_SIZE)

        crop_name = f"Face_{crop_index}.jpg"
        crop_path = OUTPUT_DIR / crop_name

        cv2.imwrite(str(crop_path), face_crop_resized)
        print(f"Saved: {crop_path.name}")
        crop_index += 1

print("done")

In [ ]:
from pathlib import Path

folder = Path("face_crops_backup")

num_files = sum(1 for f in folder.iterdir() if f.is_file())

print(f"Number of original files: {num_files}")

In [ ]:
import os
from collections import defaultdict

root_dir = "face_crops_persons"

person_counts = defaultdict(int)

# Total count
total_images = 0

for person_name in os.listdir(root_dir):
    person_path = os.path.join(root_dir, person_name)
    if os.path.isdir(person_path):
        images = [f for f in os.listdir(person_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        count = len(images)
        person_counts[person_name] += count
        total_images += count

print(f"Total images: {total_images}\n")
print("Distribution by person:")
for person, count in sorted(person_counts.items(), key=lambda x: -x[1]):
    print(f"  {person}: {count}")

In [ ]:
import os

folder = 'face_crops'

image_files = sorted([
    f for f in os.listdir(folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
])

for idx, old_name in enumerate(image_files, start=1):
    ext = os.path.splitext(old_name)[1]
    new_name = f"Face_{idx}{ext}"
    
    old_path = os.path.join(folder, old_name)
    new_path = os.path.join(folder, new_name)
    
    os.rename(old_path, new_path)

print(f"done {len(image_files)}")

In [ ]:
import os
from PIL import Image
import imagehash
from collections import defaultdict

folder = "face_crops_backup"

hashes = defaultdict(list)

valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')

for filename in os.listdir(folder):
    if filename.lower().endswith(valid_ext):
        filepath = os.path.join(folder, filename)
        try:
            with Image.open(filepath) as img:
                hash_val = imagehash.phash(img)
                hashes[str(hash_val)].append(filename)
        except Exception as e:
            print(f"Проблема с файлом {filename}: {e}")

print("\дубликаты:")
found = False
for hash_val, files in hashes.items():
    if len(files) > 1:
        found = True
        print(f"Хэш: {hash_val} — {len(files)} файла:")
        for f in files:
            print(f"    - {f}")
if not found:
    print("Дубликаты не найдены.")